In [1]:
import numpy as np
import pandas as pd
import joblib
import os

In [7]:
results_dir = "combined_UCE_5neuro"
cell_type_file = "obs_with_SubclassSupercluster.tsv.gz"
batch_size = 10_000

#cell_type_big_group_file = "cell_type_big_group_45M.tsv"

shuffled_indices_file = "shuffled_cellrow_indices.npy"

#subset_group_name = "Brain_NervousSystem"
#unshuffled_indices_subset_file = "unshuffled_cellrow_indices_Brain_NervousSystem.npy"

## CREATE BALANCED SHUFFLED BATCHES

In [3]:
# read dataset_id
dataset_ids = pd.read_csv(os.path.join(results_dir, "dataset_ids.tsv.gz"), sep="\t")

In [8]:
dataset_ids.shape

(6863203, 2)

In [ ]:
# Group row indices by dataset
row_to_dataset = dataset_ids["dataset_id"]
dataset_groups = row_to_dataset.groupby(row_to_dataset).groups  # dict: {ds_id: row_indices}

In [ ]:
row_indices = []

# Estimate how many rows per dataset should go into each batch
min_dataset_size = min(len(idx_list) for idx_list in dataset_groups.values())
n_batches = int(np.ceil(dataset_ids.shape[0] / batch_size))
min_dataset_size, n_batches

# Interleave rows from datasets
from collections import deque

# Convert index lists to deques (for fast pops from the left)
dataset_queues = {k: deque(v) for k, v in dataset_groups.items()}
print(len(dataset_queues), "datasets")

while True:
    added = 0
    chunk = []
    for ds_queue in dataset_queues.values():
        for _ in range(batch_size // len(dataset_queues)):
            if ds_queue:
                chunk.append(ds_queue.popleft())
                added += 1
    if added == 0:
        break
    
    # Shuffle the chunk before adding to row_indices
    np.random.shuffle(chunk)
    row_indices.extend(chunk)

In [8]:
row_indices = np.array(row_indices) # convert to numpy array for very fast access
len(row_indices)

45299467

In [9]:
# Save row_indices to .npy file
np.save(os.path.join(results_dir, "shuffled_cellrow_indices.npy"), row_indices)

## LOAD BALANCED SHUFFLED BATCHES back

In [3]:
shuffled_row_indices = np.load(os.path.join(results_dir, shuffled_indices_file))
shuffled_row_indices.shape

(45299467,)

## CREATE BALANCED SHUFFLED BATCHES of non malignant cells

In [10]:
# read cell_type
cell_type = pd.read_csv(os.path.join(results_dir, "cell_type.tsv.gz"), sep="\t")

In [ ]:
malignant_mask = cell_type["cell_type"].str.contains("malignant", case=False, na=False)

In [20]:
num_true = np.count_nonzero(malignant_mask)   # counts True
num_false = malignant_mask.size - num_true    # rest are False

print("True:", num_true, "False:", num_false)

True: 743748 False: 44555719


In [ ]:
row_indices_non_malignant = shuffled_row_indices[~malignant_mask[shuffled_row_indices]]

In [27]:
row_indices_non_malignant.shape, row_indices.shape[0] - row_indices_non_malignant.shape[0]

(44555719,)

In [30]:
# Save row_indices_non_malignant to .npy file
np.save(os.path.join(results_dir, "shuffled_cellrow_indices_non_malignant.npy"), row_indices_non_malignant)

## Create BALANCED SHUFFLED BATCHES of Big Groups and unshuffled indices
 {'Brain_NervousSystem'
  'Immune_Blood'
  'Other')

In [4]:
# read cell_type
cell_type = pd.read_csv(os.path.join(results_dir, cell_type_file), sep="\t")

In [5]:
# read big group file back
big_group = pd.read_csv(os.path.join(results_dir, cell_type_big_group_file), header=None, sep="\t", 
                        names=['count', 'cell_type', "big_group1", "big_group2"])
big_group

,count,cell_type,big_group1,big_group2
0,74844,amacrine cell,Brain_NervousSystem,NaN
1,6160,anterior lens cell,Brain_NervousSystem,NaN
2,727731,astrocyte,Brain_NervousSystem,NaN
3,179144,astrocyte of the cerebral cortex,Brain_NervousSystem,NaN
4,13416,Bergmann glial cell,Brain_NervousSystem,NaN
...,...,...,...,...
738,80,primary cultured cell,NaN,NaN
739,90801,progenitor cell,NaN,NaN
740,2785,somatic cell,NaN,NaN
741,66593,stem cell,NaN,NaN


In [6]:
# process big_group
cell_types_by_group1 = (
    big_group.groupby("big_group1")["cell_type"]
    .apply(list)       # turn into a list of cell_types
    .reset_index()
)


cell_types_by_group2 = (
    big_group.groupby("big_group2")["cell_type"]
    .apply(list)       # turn into a list of cell_types
    .reset_index()
)

In [7]:
big_group1 = cell_types_by_group1.rename(columns={"big_group1": "big_group", "cell_type": "cell_types1"})
big_group2 = cell_types_by_group2.rename(columns={"big_group2": "big_group", "cell_type": "cell_types2"})

# Merge on 'big_group' using outer join to keep all groups
merged_groups = pd.merge(big_group1, big_group2, on="big_group", how="outer")
merged_groups["cell_types1"] = merged_groups["cell_types1"].apply(lambda x: x if isinstance(x, list) else [])
merged_groups["cell_types2"] = merged_groups["cell_types2"].apply(lambda x: x if isinstance(x, list) else [])

merged_groups["cell_types"] = merged_groups["cell_types1"] + merged_groups["cell_types2"]

merged_groups["cell_types"].apply(len), merged_groups["cell_types1"].apply(len), merged_groups["cell_types2"].apply(len)

(0    121
 1    224
 2    394
 Name: cell_types, dtype: int64,
 0    108
 1    224
 2    394
 Name: cell_types1, dtype: int64,
 0    13
 1     0
 2     0
 Name: cell_types2, dtype: int64)

In [8]:
merged_groups

,big_group,cell_types1,cell_types2,cell_types
0,Brain_NervousSystem,"[amacrine cell, anterior lens cell, astrocyte,...","[central nervous system macrophage, mature mic...","[amacrine cell, anterior lens cell, astrocyte,..."
1,Immune_Blood,"[central nervous system macrophage, mature mic...",[],"[central nervous system macrophage, mature mic..."
2,Other,"[brain vascular cell, cardiac neuron, cerebral...",[],"[brain vascular cell, cardiac neuron, cerebral..."


In [9]:
# Dictionary to store row indices for each big_group
shuffled_row_indices_by_group = {}
unshuffled_row_indices_by_group = {}

for _, row in merged_groups.iterrows():
    group_name = row["big_group"]
    cell_list = row["cell_types"]
    
    # Boolean mask for this group
    mask = cell_type["cell_type"].isin(cell_list)
    # unshuffled indices
    unshuffled_indices = cell_type.index[mask]
    
    #shuffled indices
    shuffled_indices = shuffled_row_indices[mask[shuffled_row_indices]]
    
    # Store the indices of matching rows
    shuffled_row_indices_by_group[group_name] = shuffled_indices
    unshuffled_row_indices_by_group[group_name] = unshuffled_indices
    
group_lengths = {group: len(indices) for group, indices in shuffled_row_indices_by_group.items()}
shuffled_row_indices_by_group, unshuffled_row_indices_by_group, group_lengths

({'Brain_NervousSystem': array([29306183,  7204348,  7851231, ..., 39474539, 39475451, 39475860]),
  'Immune_Blood': array([43276780, 44910038,  1775271, ..., 39484417, 39484404, 39484440]),
  'Other': array([43677889, 10545771, 24928243, ..., 39484685, 39484492, 39484582])},
 {'Brain_NervousSystem': Index([   10000,    10001,    10002,    10003,    10004,    10005,    10006,
            10007,    10008,    10009,
         ...
         45286292, 45286293, 45286294, 45286295, 45286296, 45286297, 45286298,
         45286299, 45286300, 45286301],
        dtype='int64', length=13991888),
  'Immune_Blood': Index([       0,        1,        2,        3,        4,        5,        6,
                7,        8,        9,
         ...
         45299457, 45299458, 45299459, 45299460, 45299461, 45299462, 45299463,
         45299464, 45299465, 45299466],
        dtype='int64', length=17674247),
  'Other': Index([    1642,     6474,     9026,    10110,    10190,    10191,    10192,
            10

In [34]:
# Save row_indices_non_malignant to .npy file
for group_name in shuffled_row_indices_by_group:
    indices = shuffled_row_indices_by_group[group_name]
    np.save(os.path.join(results_dir, f"shuffled_cellrow_indices_{group_name}.npy"), indices)

for group_name in unshuffled_row_indices_by_group:
    indices = unshuffled_row_indices_by_group[group_name]
    np.save(os.path.join(results_dir, f"unshuffled_cellrow_indices_{group_name}.npy"), indices)

## Create unshuffled subset metadata files
using the unshuffled version

In [10]:
# load indices back
row_indices = np.load(os.path.join(results_dir, unshuffled_indices_subset_file))
row_indices.shape, unshuffled_indices_subset_file

((13991888,), 'unshuffled_cellrow_indices_Brain_NervousSystem.npy')

In [ ]:
'''
# build selected indices
n_rows = row_indices.shape[0]
selected_indices=[]

for i in range(0, n_rows, batch_size):
    idx_batch = row_indices[i:i + batch_size]
    idx_batch_sorted = np.sort(idx_batch)
    selected_indices.extend(idx_batch_sorted)
len(selected_indices)
'''

In [11]:
# cell_type
data = pd.read_csv(os.path.join(results_dir, "cell_type.tsv.gz"), sep="\t")
data.iloc[row_indices].to_csv(os.path.join(results_dir, f"cell_type_{subset_group_name}.tsv.gz"), sep="\t", index=False)

In [38]:
# assay
data = pd.read_csv(os.path.join(results_dir, "assay.tsv.gz"), sep="\t")
data.iloc[row_indices].to_csv(os.path.join(results_dir, f"assay_{subset_group_name}.tsv.gz"), sep="\t", index=False)

In [39]:
# tissue
data = pd.read_csv(os.path.join(results_dir, "tissue.tsv.gz"), sep="\t")
data.iloc[row_indices].to_csv(os.path.join(results_dir, f"tissue_{subset_group_name}.tsv.gz"), sep="\t", index=False)

In [40]:
# suspension_type
data = pd.read_csv(os.path.join(results_dir, "suspension_type.tsv.gz"), sep="\t")
data.iloc[row_indices].to_csv(os.path.join(results_dir, f"suspension_type_{subset_group_name}.tsv.gz"), sep="\t", index=False)

In [41]:
# dataset_ids
data = pd.read_csv(os.path.join(results_dir, "dataset_ids.tsv.gz"), sep="\t")
data.iloc[row_indices].to_csv(os.path.join(results_dir, f"dataset_ids_{subset_group_name}.tsv.gz"), sep="\t", index=False)